# 05 — Strong baselines before FNO tuning

The primary null model is **persistence**. A 7-day SST forecast can look impressive in absolute RMSE while adding little information beyond the last observed field.

This notebook establishes increasingly difficult alternatives:

1. train-only seasonal climatology;
2. persistence;
3. linear trend extrapolation;
4. EOF+ridge global low-rank linear dynamics;
5. a small local CNN (optional trained baseline).

The EOF+ridge model is particularly important. It asks whether global spatial dynamics are already captured by a simple linear model in empirical orthogonal-function coordinates.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from oisst_fno.baselines import EOFRidgeForecaster, seasonal_climatology
from oisst_fno.data import ForecastSpec, forecast_target_times, open_oisst, temporal_split
from oisst_fno.metrics import anomaly_correlation, daily_rmse, mae, rmse, skill_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path = sorted((ROOT / "data" / "raw").glob("oisst_*_ne_atlantic.nc"))[-1]
sst = open_oisst(path)["sst"]

TRAIN_END = "2024-12-31"
VALIDATION_END = "2025-12-31"
SPEC = ForecastSpec(lookback_days=14, horizon_days=7)
train_da, val_da, _ = temporal_split(sst, TRAIN_END, VALIDATION_END)

train = train_da.values.astype(float)
validation = val_da.values.astype(float)
mask = np.all(np.isfinite(train), axis=0)
target_times = forecast_target_times(val_da.time.values, SPEC)

In [ ]:
def forecast_cases(values: np.ndarray, spec: ForecastSpec):
    """Yield raw-Celsius history/target pairs for one contiguous split."""
    n_cases = len(values) - spec.lookback_days - spec.horizon_days + 1
    for start in range(n_cases):
        stop = start + spec.lookback_days
        target_index = stop + spec.horizon_days - 1
        yield values[start:stop], values[target_index]

truths = []
persistence = []
trend = []

for history, target in forecast_cases(validation, SPEC):
    truths.append(target)
    persistence.append(history[-1])
    slope = (history[-1] - history[0]) / max(SPEC.lookback_days - 1, 1)
    trend.append(history[-1] + SPEC.horizon_days * slope)

truths = np.asarray(truths)
persistence = np.asarray(persistence)
trend = np.asarray(trend)
seasonal = seasonal_climatology(
    train,
    train_da.time.values,
    target_times,
    half_window_days=15,
)
mask_3d = np.broadcast_to(mask, truths.shape)

## EOF+ridge hyperparameter selection

The EOF basis and ridge dynamics are fitted **only on training data**. The small grid below is selected by validation RMSE.

This is deliberately a serious baseline, not a straw man. It can model global spatial interactions through low-rank EOF coordinates while remaining linear and interpretable.

In [ ]:
candidate_rows = []
candidate_models = {}

for n_components in (16, 32):
    for alpha in (0.1, 1.0, 10.0):
        candidate = EOFRidgeForecaster(n_components=n_components, alpha=alpha).fit(train, SPEC)
        prediction = candidate.predict_series(validation)
        score = rmse(prediction, truths, mask_3d)
        candidate_rows.append(
            {"n_components": n_components, "alpha": alpha, "validation_rmse_c": score}
        )
        candidate_models[(n_components, alpha)] = prediction

eof_search = pd.DataFrame(candidate_rows).sort_values("validation_rmse_c")
eof_search

In [ ]:
best = eof_search.iloc[0]
best_key = (int(best["n_components"]), float(best["alpha"]))
eof_ridge = candidate_models[best_key]

persistence_rmse = rmse(persistence, truths, mask_3d)
rows = []
for name, prediction in {
    "seasonal_climatology": seasonal,
    "persistence": persistence,
    "linear_trend": trend,
    "eof_ridge": eof_ridge,
}.items():
    model_rmse = rmse(prediction, truths, mask_3d)
    rows.append(
        {
            "model": name,
            "rmse_c": model_rmse,
            "mae_c": mae(prediction, truths, mask_3d),
            "anomaly_correlation": anomaly_correlation(prediction, truths, seasonal, mask_3d),
            "persistence_skill": skill_score(model_rmse, persistence_rmse),
        }
    )

results = pd.DataFrame(rows).sort_values("rmse_c")
results

In [ ]:
validation_persistence_daily = daily_rmse(persistence, truths, mask_3d)
hard_persistence_threshold_c = float(np.quantile(validation_persistence_daily, 0.75))

selection = {
    "train_end": TRAIN_END,
    "validation_end": VALIDATION_END,
    "lookback_days": SPEC.lookback_days,
    "horizon_days": SPEC.horizon_days,
    "seasonal_climatology_half_window_days": 15,
    "eof_ridge": {"n_components": best_key[0], "alpha": best_key[1]},
    "validation_persistence_daily_rmse_q75_c": hard_persistence_threshold_c,
    "metrics": rows,
    "eof_search": candidate_rows,
}

metrics_path = ROOT / "artifacts" / "metrics" / "baseline_selection.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(selection, indent=2), encoding="utf-8")
results.to_csv(ROOT / "artifacts" / "metrics" / "baselines_validation.csv", index=False)
print("hard-persistence threshold from validation:", hard_persistence_threshold_c)
print(metrics_path)

## Optional learned local baseline

A small CNN asks whether learned **local spatial filtering** is already sufficient.

It should use the exact same train/validation windows and tuning policy as the FNO. Do not give the FNO a much larger tuning budget and then interpret the comparison as architectural evidence.

For the research-grade benchmark, extend this to a ConvLSTM or U-Net-style spatiotemporal model as specified in `prompts/07_research_grade_benchmark.md`.

In [ ]:
import torch
from torch import nn


class SmallCNN(nn.Module):
    """Minimal local convolutional baseline for the same lookback stack."""

    def __init__(self, in_channels: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 1, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# Train and persist this result before making paper-level claims about FNO superiority.